> **LangChain 1.x note:** This notebook uses the current package layout (`langchain_huggingface`, `langchain_openai`, `langchain_core`). It runs locally with open embedding models; API-based embeddings are optional and gated behind a config flag.

## Chapter 2 — Embedding Evaluation for Science (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare)

### Learning objectives
- Build a small labeled scientific retrieval set (query → relevant documents).
- Compute **recall@k** and **MRR** to compare embedding models.
- Run a qualitative error analysis on retrieval failures.
- Reason about the dimension-vs-cost-vs-quality trade-off.

**Runtime / cost:** CPU, ~5–8 min for open models. **Data:** synthetic biomedical snippets (no external corpus).

## Why evaluate embeddings on *your* data

Leaderboard scores (MTEB) rarely transfer directly to a niche scientific domain. A model that wins on web text may fail on gene symbols, drug names, or assay jargon. The only reliable test is a **small labeled retrieval set drawn from your own domain**. This notebook shows the full loop: build → embed → rank → score → inspect failures.

### API / credentials

Open embedding models run locally (no key). To also score a hosted embedding model (OpenAI / Gemini / etc.), set `RUN_API_EMBEDDINGS=True` in the config cell and provide a key via Colab Secrets or a local `.env`.

In [ ]:
import os

try:
    from google.colab import userdata  # type: ignore
    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False

if not IN_COLAB:
    try:
        from dotenv import load_dotenv; load_dotenv()
    except Exception:
        pass

def get_secret(name, default=None):
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val: return val
        except Exception: pass
    return os.getenv(name, default)

API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"
if API_KEY_PROVIDER == "OPENAI": os.environ["OPENAI_API_KEY"] = get_secret("LC4LSH_OPENAI_API_KEY", "sk-...")
elif API_KEY_PROVIDER == "ANTHROPIC": os.environ["ANTHROPIC_API_KEY"] = get_secret("LC4LSH_ANTHROPIC_API_KEY", "sk-ant-...")
elif API_KEY_PROVIDER == "GEMINI": os.environ["GOOGLE_API_KEY"] = get_secret("LC4LSH_GOOGLE_API_KEY", "AIza...")
elif API_KEY_PROVIDER == "GROQ": os.environ["GROQ_API_KEY"] = get_secret("LC4LSH_GROQ_API_KEY", "gsk_...")
print(f"✅ API keys loaded for {API_KEY_PROVIDER} (source: {'Colab Secrets' if IN_COLAB else 'local env/.env'})")
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""

In [ ]:
import os

SEED = 42
RUN_API_EMBEDDINGS = False  # set True to also score a hosted embedding model
OPEN_MODELS = [
    "sentence-transformers/all-MiniLM-L6-v2",      # small, fast, 384-d
    "sentence-transformers/all-mpnet-base-v2",      # stronger general, 768-d
]
API_EMBEDDING_MODEL = "text-embedding-3-large"  # used only if RUN_API_EMBEDDINGS
RUN_METADATA = {"chapter": 2, "notebook": "embedding_evaluation", "seed": SEED}
print(RUN_METADATA)

## Package Installation and Setup

Pinned versions with upper bounds — Last validated: 2026-07-21 (see UPDATE_2026.md).

In [ ]:
%pip install -q "sentence-transformers>=3.0,<6" "langchain-huggingface>=0.1" "langchain-openai>=0.2" numpy pandas scikit-learn

## 1. A tiny labeled retrieval set

Queries with one or more known-relevant documents. In practice you'd build this from real search logs or expert annotation; here it's synthetic but domain-flavored (drug–target, gene–disease, assay).

In [ ]:
DOCS = {
    "d1": "Imatinib is a tyrosine kinase inhibitor targeting BCR-ABL, used in chronic myeloid leukemia.",
    "d2": "Aspirin irreversibly acetylates cyclooxygenase-1, reducing thromboxane A2 and platelet aggregation.",
    "d3": "BRCA1 and BRCA2 mutations increase lifetime risk of breast and ovarian cancer.",
    "d4": "Surface plasmon resonance measures binding affinity (KD) between a small molecule and its protein target.",
    "d5": "Metformin activates AMP-activated protein kinase (AMPK) and is first-line therapy for type 2 diabetes.",
    "d6": "TP53 is a tumor suppressor; loss-of-function mutations are found in over half of human cancers.",
    "d7": "The IC50 is the concentration of inhibitor required to reduce enzyme activity by 50 percent.",
    "d8": "CRISPR-Cas9 enables targeted genome editing via guide RNA-directed double-strand breaks.",
}

# query -> list of relevant doc ids
QUERIES = {
    "q1": ("which drug inhibits BCR-ABL in CML?", ["d1"]),
    "q2": ("how does aspirin affect platelets?", ["d2"]),
    "q3": ("genes linked to hereditary breast cancer", ["d3"]),
    "q4": ("method to measure drug-protein binding affinity", ["d4"]),
    "q5": ("tumor suppressor mutated in most cancers", ["d6"]),
    "q6": ("what does IC50 mean in an enzyme assay?", ["d7"]),
}
print(f"{len(DOCS)} docs, {len(QUERIES)} queries")

## 2. Metrics

**Recall@k**: fraction of queries whose relevant doc appears in the top-k. **MRR** (mean reciprocal rank): average of `1/rank` of the first relevant hit.

In [ ]:
import numpy as np

def rank_docs(query_emb, doc_embs, doc_ids):
    sims = doc_embs @ query_emb / (np.linalg.norm(doc_embs, axis=1) * np.linalg.norm(query_emb) + 1e-9)
    order = np.argsort(-sims)
    return [doc_ids[i] for i in order]

def recall_at_k(ranked, relevant, k):
    return 1.0 if any(d in relevant for d in ranked[:k]) else 0.0

def reciprocal_rank(ranked, relevant):
    for i, d in enumerate(ranked):
        if d in relevant:
            return 1.0 / (i + 1)
    return 0.0

def evaluate(embed_fn, label):
    doc_ids = list(DOCS.keys())
    doc_embs = embed_fn([DOCS[d] for d in doc_ids])
    ks = [1, 3]
    recalls = {k: [] for k in ks}
    rrs, failures = [], []
    for qid, (qtext, rel) in QUERIES.items():
        q_emb = embed_fn([qtext])[0]
        ranked = rank_docs(q_emb, doc_embs, doc_ids)
        for k in ks:
            recalls[k].append(recall_at_k(ranked, rel, k))
        rrs.append(reciprocal_rank(ranked, rel))
        if ranked[0] not in rel:
            failures.append((qid, qtext, rel, ranked[:3]))
    out = {"model": label}
    for k in ks:
        out[f"recall@{k}"] = round(float(np.mean(recalls[k])), 3)
    out["MRR"] = round(float(np.mean(rrs)), 3)
    return out, failures

## 3. Score open embedding models

In [ ]:
from sentence_transformers import SentenceTransformer

results, all_failures = [], {}
for name in OPEN_MODELS:
    try:
        m = SentenceTransformer(name)
        fn = lambda texts: np.asarray(m.encode(texts, normalize_embeddings=True))
        row, fails = evaluate(fn, name.split("/")[-1])
        row["dim"] = m.get_sentence_embedding_dimension()
        results.append(row)
        all_failures[name] = fails
        print(f"✅ {name}: {row}")
    except Exception as e:
        print(f"⚠️  skipped {name}: {type(e).__name__}: {str(e)[:80]}")

In [ ]:
if RUN_API_EMBEDDINGS:
    try:
        from langchain_openai import OpenAIEmbeddings
        api_emb = OpenAIEmbeddings(model=API_EMBEDDING_MODEL)
        fn = lambda texts: np.asarray(api_emb.embed_documents(texts))
        row, fails = evaluate(fn, API_EMBEDDING_MODEL)
        row["dim"] = len(fn(["probe"])[0])
        results.append(row)
        all_failures[API_EMBEDDING_MODEL] = fails
        print(f"✅ {API_EMBEDDING_MODEL}: {row}")
    except Exception as e:
        print(f"⚠️  API embeddings failed: {type(e).__name__}: {str(e)[:80]}")
else:
    print("RUN_API_EMBEDDINGS=False — skipping hosted embedding model.")

In [ ]:
import pandas as pd
res_df = pd.DataFrame(results)
res_df

## 4. Qualitative error analysis

Numbers alone mislead. Inspect the queries each model got wrong and look for patterns (abbreviations, gene symbols, numeric assay terms).

In [ ]:
for model, fails in all_failures.items():
    print(f"\n=== {model}: {len(fails)} top-1 failures ===")
    for qid, qtext, rel, top3 in fails:
        print(f"  [{qid}] '{qtext}'\n      expected={rel}  got top3={top3}")

## 5. Dimension vs cost

Higher-dimensional embeddings cost more to store and search. `all-MiniLM-L6-v2` (384-d) is ~2× cheaper to index than `all-mpnet-base-v2` (768-d) per vector; hosted models add per-token cost. Plot the quality-vs-dimension trade-off.

In [ ]:
import matplotlib.pyplot as plt

if len(res_df):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(res_df["dim"], res_df["MRR"], s=80)
    for _, r in res_df.iterrows():
        ax.annotate(r["model"], (r["dim"], r["MRR"]), textcoords="offset points", xytext=(6, 4), fontsize=8)
    ax.set_xlabel("embedding dimension (proxy for storage/search cost)")
    ax.set_ylabel("MRR (quality)")
    ax.set_title("Quality vs cost trade-off")
    plt.tight_layout(); plt.show()

## Limitations & safety
- This retrieval set is **tiny and synthetic** — use it to learn the method, not to draw conclusions. Real evaluation needs hundreds of queries.
- Recall@k on 6 queries has huge variance; report confidence intervals on real data.
- A model can win recall@1 yet still retrieve near-duplicates or leaky text; inspect failures qualitatively.
- Hosted embeddings change over time; pin model versions for reproducibility.

## Cleanup
Release models and free memory.

In [ ]:
import gc
try:
    del m
except Exception:
    pass
gc.collect()
print("🧹 embedding models released")

## Exercises

1. Why can a model with a higher MTEB score underperform on your biomedical retrieval set?
   <details><summary>Hint</summary>Domain shift: gene symbols, drug names, and assay jargon are rare in general web corpora used for benchmark training.</details>
2. What does a top-1 failure where the model returns a *chemically similar but wrong* doc tell you?
   <details><summary>Hint</summary>The embedding captures semantic similarity but not the precise entity relation; consider a cross-encoder reranker.</details>
3. When is a smaller, lower-quality embedding the right choice?
   <details><summary>Hint</summary>At very large scale or tight latency budgets, a cheaper first-stage retriever + reranker can beat one expensive model.</details>

### Task A — Add a biomedical embedder
Add a domain model (e.g., `pritamdeka/S-PubMedBert-MS-MARCO` or `BAAI/bge-base-en`). Does it beat the general models on these biomedical queries?

### Task B — Reranking
Take the top-5 from the best bi-encoder and re-score with a cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`). Measure the recall@1 improvement.

### Task C — Build a real mini-set
Write 10 queries + relevant docs from a paper you know. Re-run `evaluate`. Which failure modes appear that the synthetic set missed?

### Task D — Cost model
For a 1M-document corpus, estimate storage (float32) and search-cost differences between a 384-d and a 3072-d embedding. At what corpus size does the cheaper model win even at slightly lower recall?